# 03. Query Rewriting (쿼리 재작성) - AI 트렌드 보고서 기반

## 왜 필요한가?

사용자 질문이 **모호하거나 구어체**이면 벡터 검색 품질이 떨어질 수 있습니다.  
AI 트렌드 보고서는 `AI 산업`, `AI 기술`, `AI 정책`, `2026년 전망`, `토픽모델링`, `추론형 데이터`처럼 보고서 내부 용어가 비교적 명확합니다.

따라서 사용자의 일상 표현을 보고서 검색에 적합한 용어로 바꾸면 검색 품질을 개선할 수 있습니다.

```text
원본 질문:     "내년 AI는 어떻게 바뀌어요?"
재작성 질문:   "2026년 AI 산업·기술·정책 전망에서 핵심 변화는 무엇인가요?"

원본 질문:     "AI가 회사 일에 어떻게 들어와요?"
재작성 질문:   "2026년 AI 산업 전망에서 기업 운영 인프라와 AI 에이전트 도입은 어떻게 변화할 것으로 전망되나요?"

원본 질문:     "AI 규제는 왜 중요해졌나요?"
재작성 질문:   "AI 정책 토픽에서 안전성, 책임성, 규제 거버넌스가 중요한 이슈로 부각된 이유는 무엇인가요?"
```

## 접근 방법

1. **Query Rewriting** - 구어체·모호한 질문을 보고서 검색에 적합한 질의로 변환
2. **HyDE** - 질문에 대한 가상 답변을 생성한 뒤 해당 텍스트로 검색
3. **RAG Chain** - 재작성된 질문으로 검색하고, 검색 문서를 바탕으로 답변 생성


## 실행 전 준비

이 노트북은 `02_data_pdf_to_md.ipynb`에서 생성한 Markdown 기반 ChromaDB를 사용한다.  
따라서 먼저 02번 노트북을 실행하여 아래 폴더가 생성되어 있어야 한다.

```text
chroma_db/ai_trend_report_md
```

확인할 사항은 다음과 같다.

1. `data/AI@Data_Report_CLEANED.md` 파일이 생성되어 있는지 확인한다.
2. `chroma_db/ai_trend_report_md` 폴더가 생성되어 있는지 확인한다.
3. 현재 노트북은 PDF 직접 로딩 기반 DB가 아니라 Markdown 기반 DB를 사용한다.
4. Query Rewriting은 원본 질문의 의도를 유지하면서 검색에 적합한 표현으로 바꾸는 것이 핵심이다.


In [ ]:
from pathlib import Path

from dotenv import load_dotenv
load_dotenv(override=True, dotenv_path="../.env")

from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_chroma import Chroma
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

# 02_data_pdf_to_md.ipynb에서 생성한 Markdown 기반 ChromaDB를 불러온다.

DB_PATH = Path("chroma_db/ai_trend_report_md")
COLLECTION_NAME = "ai_trend_report_2026_md"

vector_store = Chroma(
    embedding_function=embeddings,
    collection_name=COLLECTION_NAME,
    persist_directory=DB_PATH
)

retriever = vector_store.as_retriever(search_kwargs={"k": 5})

print(f"불러온 청크 수: {vector_store._collection.count()}")


불러온 청크 수: 29


- Top-k 검색 함수 정의

In [2]:
def inspect_retrieval(query, k=5):
    """검색 결과를 distance score와 함께 확인한다.
    ChromaDB의 score는 거리값이므로 낮을수록 더 유사하다.
    """
    results = vector_store.similarity_search_with_score(query, k=k)

    print(f"[질문] {query}")
    print("-" * 80)
    print("ChromaDB distance score: 낮을수록 더 유사함")

    for i, (doc, score) in enumerate(results, start=1):
        metadata = doc.metadata
        source = metadata.get("source", "?")
        chunk_id = metadata.get("chunk_id", "?")
        doc_type = metadata.get("doc_type", "?")
        source_name = metadata.get("source_name", "?")

        print(f"[{i}] distance={score:.4f} | chunk_id={chunk_id} | doc_type={doc_type}")
        print(f"source={source}")
        print(f"source_name={source_name}")
        print(doc.page_content[:500])  # 내용 일부만 출력
        print("-" * 80)

## 1. 기본 Query Rewriting

In [ ]:
# 재작성 프롬프트 구성
rewrite_prompt = ChatPromptTemplate.from_messages([
    ("system", """당신은 AI 트렌드 보고서 기반 RAG 검색 쿼리 개선 전문가입니다.
사용자의 구어체 또는 모호한 질문을 AI 트렌드 보고서 검색에 적합한 명확한 질의로 재작성하세요.

재작성 시 다음 보고서 용어를 적절히 활용하세요.
- AI 산업, AI 기술, AI 정책
- 2026년 전망
- 토픽모델링, 주요 키워드, 핵심 이슈
- AI 에이전트, 멀티모달, 추론형 AI, 온디바이스 AI
- 안전성, 책임성, 규제 거버넌스, AI 기본법

재작성된 쿼리만 출력하고 설명은 하지 마세요."""),
    ("human", "원본 질문: {question}\n\n재작성 질문:")
])

# 재작성 체인 구성
rewrite_chain = rewrite_prompt | llm | StrOutputParser()

# 질문 테스트
original_queries = [
    "내년 AI는 어떻게 바뀌어요?",
    "AI가 회사 일에 어떻게 들어와요?",
    "AI 규제는 왜 중요해졌나요?",
    "온디바이스 AI가 뭔 의미예요?",
    "추론형 데이터는 어떤 종류가 있어요?"
]

# 재작성된 질문 출력
for q in original_queries:
    rewritten = rewrite_chain.invoke({"question": q})
    print(f"원본: {q}")
    print(f"재작성: {rewritten}")
    print()


원본: 내년 AI는 어떻게 바뀌어요?
재작성: 2026년 전망에 따른 AI 산업의 변화와 주요 키워드는 무엇인가요?

원본: AI가 회사 일에 어떻게 들어와요?
재작성: AI 산업에서 AI 기술이 회사 업무에 어떻게 통합되고 있는지에 대한 주요 키워드와 핵심 이슈는 무엇인가요? 2026년 전망을 포함하여 설명해 주세요.

원본: AI 규제는 왜 중요해졌나요?
재작성: AI 정책의 중요성이 증가한 이유와 관련된 핵심 이슈는 무엇인가요?

원본: 온디바이스 AI가 뭔 의미예요?
재작성: 온디바이스 AI의 의미와 AI 산업 내에서의 역할에 대해 설명해 주세요.

원본: 추론형 데이터는 어떤 종류가 있어요?
재작성: 추론형 AI와 관련된 데이터의 종류와 그 활용에 대한 주요 키워드는 무엇인가요?



### 확인 포인트
원본 질문은 구어체 표현과 상대 시점 표현을 포함한다.  
재작성 질문은 보고서의 용어인 `2026년 전망`, `AI 기술`, `AI 모델`, `핵심 변화`와 더 가까운 표현으로 바뀐다.

검색 결과를 비교할 때는 score 값만 보지 말고, 검색된 청크가 질문 의도와 얼마나 직접적으로 연결되는지 확인한다.

## 2. 재작성 전후 검색 결과 비교

In [ ]:
original = "내년 AI 모델은 어떻게 발전해?"
rewritten = rewrite_chain.invoke({"question": original})

# 원본 쿼리 검색 결과
print(f"[원본 쿼리] {original}")
inspect_retrieval(original, k=3)

# 재작성된 쿼리 검색 결과
print(f"\n[재작성 쿼리] {rewritten}")
inspect_retrieval(rewritten, k=3)


[원본 쿼리] 내년 AI 모델은 어떻게 발전해?
[질문] 내년 AI 모델은 어떻게 발전해?
--------------------------------------------------------------------------------
ChromaDB distance score: 낮을수록 더 유사함
[1] distance=0.9062 | chunk_id=6 | doc_type=markdown
source=data\AI@Data_Report_CLEANED.md
source_name=AI@Data_Report_CLEANED.md
- - 산업·정책 영역에서는 제도화 지연에 따른 불확실성과 대응 부담이 증가하고 있으며, 국가·기업 모두 AI 활용과 규제 준수 사이에서 복합적 리스크에 직면하고 있음 

> 1) Stanford HAI, 2025 AI Index Report – Economy 

> 2) OECD AI Incidents Monitor(AIM) 

## **1.2 분석 목적** 

- 본 이슈페이퍼는 2025년을 전후해 급격히 재편되는 글로벌 AI 재편 흐름을 진단 하기 위해, 산업·기술·정책 전반에 반복적으로 등장하는 핵심 키워드와 국제 담론을 종합적으로 수집·분석하는 것을 목적으로 함 

- - 2025년을 분석 기준 시점으로 설정함으로써, 2023~2024년의 실험·도입 국면에서 2025년 이후 전면 확산·제도화 국면으로의 전환이 어떻게 나타나는지를 계량적으로 비교·검증하고자 함 

- - 단순 기사 검토 수준을 넘어 글로벌 싱크탱크·정책 보고서·산업 리포트 등 다층적 자료를 기반으로 하여 AI
--------------------------------------------------------------------------------
[2] distance=0.9461 | chunk_id=23 | doc_type=markdown
source=data\AI@Data_Report_CLEANED.md
source_name=AI@Data_Report_CL

### Query Rewriting과 HyDE의 차이

Query Rewriting과 HyDE는 모두 검색 품질을 높이기 위한 방법이지만, 생성하는 중간 표현이 다르다.

| 구분 | Query Rewriting | HyDE |
|---|---|---|
| 목적 | 질문을 검색에 적합한 질의로 재작성 | 질문에 대한 가상의 답변 문서를 생성 |
| 입력 | 사용자 질문 | 사용자 질문 |
| 출력 | 짧고 명확한 검색 질문 | 답변 형태의 긴 검색용 문서 |
| 검색 기준 | 재작성된 질문 | 가상 문서 |
| 주의점 | 원본 질문의 의도를 바꾸면 안 됨 | 가상 문서가 실제 문서 내용과 다를 수 있음 |

즉, Query Rewriting은 “질문을 다시 쓰는 방식”이고, HyDE는 “검색용 가상 문서를 만드는 방식”이다.  
HyDE에서 생성한 문장은 최종 답변이 아니라 검색 품질을 높이기 위한 중간 표현으로 이해해야 한다.


## 3. HyDE (Hypothetical Document Embeddings)

질문에 대한 **가상의 답변**을 먼저 생성 → 그 답변으로 검색

In [5]:
hyde_prompt = ChatPromptTemplate.from_messages([
    ("system", """당신은 AI 트렌드 보고서 분석가입니다.
주어진 질문에 대해 보고서에 포함되어 있을 법한 설명문을 3~4문장으로 작성하세요.
AI 산업·기술·정책, 2026년 전망, 토픽모델링, 핵심 이슈 등의 용어를 자연스럽게 사용하세요.
단, 실제 답변을 확정하지 말고 검색 품질을 높이기 위한 가상 문서처럼 작성하세요."""),
    ("human", "{question}")
])

hyde_chain = hyde_prompt | llm | StrOutputParser()

question = "내년 AI 기술은 어떤 방향으로 발전할까요?"

hypothetical_doc = hyde_chain.invoke({"question": question})
print("[가상 보고서 답변]")
print(hypothetical_doc)


[가상 보고서 답변]
2026년 전망에 따르면, AI 기술은 더욱 고도화된 자율성과 적응성을 갖춘 방향으로 발전할 것으로 예상됩니다. 특히, 토픽모델링 기법의 발전은 데이터 분석의 효율성을 높이고, 다양한 산업 분야에서 맞춤형 솔루션을 제공하는 데 기여할 것입니다. 또한, AI 산업의 정책적 측면에서도 윤리적 기준과 규제가 강화되어, 기술의 안전한 활용과 사회적 수용성을 높이는 핵심 이슈로 부각될 것입니다. 이러한 변화는 AI 기술이 사회 전반에 걸쳐 긍정적인 영향을 미치는 데 중요한 역할을 할 것으로 보입니다.


In [6]:
# 가상 문서로 검색 vs 원본 질문으로 검색 비교
print("[원본 질문으로 검색]")
inspect_retrieval(question, k=3)

print("\n[가상 보고서 답변으로 검색]")
inspect_retrieval(hypothetical_doc, k=3)


[원본 질문으로 검색]
[질문] 내년 AI 기술은 어떤 방향으로 발전할까요?
--------------------------------------------------------------------------------
ChromaDB distance score: 낮을수록 더 유사함
[1] distance=0.7440 | chunk_id=6 | doc_type=markdown
source=data\AI@Data_Report_CLEANED.md
source_name=AI@Data_Report_CLEANED.md
- - 산업·정책 영역에서는 제도화 지연에 따른 불확실성과 대응 부담이 증가하고 있으며, 국가·기업 모두 AI 활용과 규제 준수 사이에서 복합적 리스크에 직면하고 있음 

> 1) Stanford HAI, 2025 AI Index Report – Economy 

> 2) OECD AI Incidents Monitor(AIM) 

## **1.2 분석 목적** 

- 본 이슈페이퍼는 2025년을 전후해 급격히 재편되는 글로벌 AI 재편 흐름을 진단 하기 위해, 산업·기술·정책 전반에 반복적으로 등장하는 핵심 키워드와 국제 담론을 종합적으로 수집·분석하는 것을 목적으로 함 

- - 2025년을 분석 기준 시점으로 설정함으로써, 2023~2024년의 실험·도입 국면에서 2025년 이후 전면 확산·제도화 국면으로의 전환이 어떻게 나타나는지를 계량적으로 비교·검증하고자 함 

- - 단순 기사 검토 수준을 넘어 글로벌 싱크탱크·정책 보고서·산업 리포트 등 다층적 자료를 기반으로 하여 AI
--------------------------------------------------------------------------------
[2] distance=0.7514 | chunk_id=22 | doc_type=markdown
source=data\AI@Data_Report_CLEANED.md
source_name=AI@Data_Report_CLEANED.md

HyDE는 검색 품질을 높일 수 있지만, 생성된 가상 문서가 실제 보고서 표현과 다르면 오히려 검색 방향이 흔들릴 수 있다.

## 4. Query Rewriting을 포함한 RAG 체인

```
원본 질문
→ 검색하기 좋은 질문으로 재작성
→ 재작성된 질문으로 Vector Store 검색
→ 검색된 문서를 context로 결합
→ 원본 질문 + 검색 문서를 LLM에 전달
→ 최종 답변 생성
```

In [7]:
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

answer_prompt = ChatPromptTemplate.from_messages([
    ("system", """아래 문서에 근거해서만 질문에 답하세요.
문서에서 확인할 수 없는 내용은 '문서에서 확인할 수 없습니다'라고 답하세요.
한줄에 한문장씩 깔끔하게 정리하세요.

[문서]
{context}"""),
    ("human", "{question}")
])

# --------------------------------------------------
# 1. 사용자 질문 입력
# --------------------------------------------------

question = "AI가 회사 일에 어떻게 들어와요?"
print("[원본 질문]")
print(question)

# --------------------------------------------------
# 2. 질문 재작성
# --------------------------------------------------
# 사용자의 질문을 검색에 더 적합한 표현으로 바꾼다.

rewritten_question = rewrite_chain.invoke({
    "question": question
})
print("\n[재작성된 질문]")
print(rewritten_question)

# --------------------------------------------------
# 3. 재작성된 질문으로 문서 검색
# --------------------------------------------------
# 원본 질문이 아니라, 재작성된 질문을 Vector Store에 검색한다.

docs = retriever.invoke(rewritten_question)

print("\n[검색된 문서 수]")
print(len(docs))

# --------------------------------------------------
# 4. 검색된 문서를 하나의 context로 합치기
# --------------------------------------------------

context = format_docs(docs)

print("\n[검색 문서 미리보기]")
print(context[:1000])


# --------------------------------------------------
# 5. 답변 프롬프트 만들기
# --------------------------------------------------
# 원본 질문과 검색된 문서를 함께 LLM에게 전달한다.

messages = answer_prompt.invoke({
    "question": question,
    "context": context
})


# --------------------------------------------------
# 6. LLM 답변 생성
# --------------------------------------------------
response = llm.invoke(messages)


# --------------------------------------------------
# 7. 문자열 결과만 추출
# --------------------------------------------------
result = StrOutputParser().invoke(response)

print("\n[최종 답변]")
print(result)

[원본 질문]
AI가 회사 일에 어떻게 들어와요?

[재작성된 질문]
AI 산업에서 AI 기술이 회사 업무에 어떻게 통합되고 있는지에 대한 주요 키워드와 핵심 이슈는 무엇인가요? 2026년 전망을 포함하여 AI 에이전트, 멀티모달, 추론형 AI의 역할을 설명해 주세요.

[검색된 문서 수]
5

[검색 문서 미리보기]
- - 이와 함께 기존 산업의 가치 사슬도 전면 재편이라기보다는 핵심 공정·고부가 서비스 단에서 데이터·AI 중심 구조로 이동하는 압력이 강화되는 양상을 보일 것으로 전망됨 

- 기업 내부에서 AI 에이전트를 활용한 문서 처리, 고객 지원, 운영 자동화 등이 증가하며 – – 

- 사람 에이전트 시스템이 혼합된 업무 구조 가 일부 영역에서 확산될 가능성이 있음 

- - 기업 간(B2B) 시스템 연동을 통해 제한된 범위에서 AI 간 협업 형태의 자동화 사례도 등장할 것으로 예상되지만, 그 영향은 아직 초기 단계에 머물며 중장기적 구조 변화의 단초를 형성하는 수준으로 평가됨 

- ※ IoT 센서와 AI 기술을 통해 물류 및 공급망 전 과정을 자동화하고 최적화[4)] 

## **[ 그림 5 ] 주요 산업 분야별 AI 적용 사례** 

## **4.3 2026년 AI 기술 전망** 

- 2026년 AI 기술은 지능 구조 고도화 와 데이터 한계 보완을 중심 으로 진화할 전망임 

- - 합성데이터·추론형 AI·멀티모달 기술이 주요 경쟁 축으로 자리 잡으면서, 학습 효율 향상·복합 정보 처리·설명 가능성 강화 등 모델 내부 구조의 질적 개선 흐름이 이어질 것으로 예상됨 

- 기술 고도화는 입력 유형과 데이터 범위를 확장함으로써 AI의 활용 가능성을 넓히고, 복잡한 상황에서의 판단 능력을 강화하는 방향 으로 진화할 것으로 전망됨 

- - 고품질 데이터 생성·멀티모달·고급 추론 기술이 결합되며 AI의 상황 이해와 문제 해결 능력이 향상되고, 서비스·산업 전반의 활용도도 확대될 전망임

- - 산업·정책 영역에서는 제도화 지연에 따른 불확실성과 대응

- Runnable 객체로 표현하기

In [ ]:
from langchain_core.runnables import RunnablePassthrough, RunnableLambda
from operator import itemgetter

rag_chain = (
    RunnablePassthrough.assign(
        # 1. 원본 질문을 검색에 적합한 질문으로 재작성
        rewritten_question=itemgetter("question") | rewrite_chain
    )
    .assign(
        # 2. 재작성된 질문으로 문서 검색
        docs=itemgetter("rewritten_question") | retriever
    )
    .assign(
        # 3. 검색된 문서를 context 문자열로 변환
        context=itemgetter("docs") | RunnableLambda(format_docs)
    )
    .assign(
        # 4. 원본 질문 + 검색 context를 기반으로 답변 생성
        answer=(
            {
                "question": itemgetter("question"),
                "context": itemgetter("context")
            }
            | answer_prompt
            | llm
            | StrOutputParser()
        )
    )
)

#### LCEL 문법으로

실행흐름
```text
question 입력
  ↓
question은 그대로 유지
  ↓
question → rewrite_chain → retriever → format_docs → context 생성
  ↓
question + context → answer_prompt
  ↓
llm
  ↓
StrOutputParser
  ↓
최종 답변
```

In [8]:
# 딕셔너리에서 특정 키의 값을 추출하는 함수
from operator import itemgetter  

from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser


def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)


answer_prompt = ChatPromptTemplate.from_messages([
    ("system", """아래 문서에 근거해서만 질문에 답하세요.
문서에서 확인할 수 없는 내용은 '문서에서 확인할 수 없습니다'라고 답하세요.
한줄에 한문장씩 깔끔하게 정리하세요.

[문서]
{context}"""),
    ("human", "{question}")
])


rewrite_rag_chain = (
    {
        "question": itemgetter("question"),
        "context": (
            {"question": itemgetter("question")}
            | rewrite_chain
            | retriever
            | format_docs
        )
    }
    | answer_prompt
    | llm
    | StrOutputParser()
)


result = rewrite_rag_chain.invoke({
    "question": "AI가 회사 일에 어떻게 들어와요?"
})

print(result)

AI는 기업 내부에서 문서 처리, 고객 지원, 운영 자동화 등에 활용됩니다.  
사람 에이전트 시스템이 혼합된 업무 구조가 일부 영역에서 확산될 가능성이 있습니다.  
기업 간 시스템 연동을 통해 AI 간 협업 형태의 자동화 사례도 등장할 것으로 예상됩니다.


#### 선택 실습: Runnable 객체 사용

이 부분은 LCEL 체인 구성을 더 명확하게 이해하기 위한 확장 예제이다.  
초보자 수업에서는 선택 실습으로 다루고, 기본 흐름은 앞의 단계별 실행과 LCEL 방식만 이해해도 충분하다.

실행흐름
```text
입력 질문
  ↓
질문 재작성(원본 question 유지 + rewritten 생성)
  ↓
재작성 질문으로 문서 검색(rewritten을 사용해 retriever 검색)
  ↓
context 생성(검색 문서를 context로 정리)
  ↓
answer_prompt 생성(question, context만 사용됨)
  ↓
llm 답변 생성
  ↓
최종 답변 출력
```

In [9]:
from langchain_core.runnables import RunnablePassthrough

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

answer_prompt = ChatPromptTemplate.from_messages([
    ("system", """아래 문서에 근거해서만 질문에 답하세요.
문서에서 확인할 수 없는 내용은 '문서에서 확인할 수 없습니다'라고 답하세요.
한줄에 한문장씩 깔끔하게 정리하세요.

[문서]
{context}"""),
    ("human", "{question}")
])

# 쿼리 재작성 → 검색 → 답변 생성
rewrite_rag_chain = (
    # 1. 원본 질문을 유지하면서, 검색에 적합한 질문으로 재작성한 결과를 추가한다.
    # --------------------------------------------------
    # 1단계. 원본 입력을 그대로 유지하면서 rewritten 값 추가
    # --------------------------------------------------
    # 입력 예:
    # {"question": "AI가 회사 일에 어떻게 들어와요?"}
    #
    # RunnablePassthrough.assign()은 기존 입력 question을 유지한 채,
    # rewritten이라는 새로운 key를 추가한다.
    #
    # rewritten에는 rewrite_chain이 생성한 "검색용 질문"이 들어간다.
    RunnablePassthrough.assign(
        rewritten=lambda x: rewrite_chain.invoke({"question": x["question"]})
    )
    # 2. 재작성된 질문으로 Vector Store에서 관련 문서를 검색하고 context로 묶는다.
    # --------------------------------------------------
    # 2단계. 재작성된 질문으로 문서 검색 후 context 생성
    # --------------------------------------------------
    # 이전 단계 결과 예:
    # {
    #   "question": "AI가 회사 일에 어떻게 들어와요?",
    #   "rewritten": "AI가 기업 업무 자동화와 운영 방식에 미치는 영향은 무엇인가?"
    # }
    #
    # x["rewritten"]을 사용해 retriever에서 관련 문서를 검색한다.
    # 검색된 문서들은 format_docs()를 통해 하나의 문자열 context로 합쳐진다.

    | RunnablePassthrough.assign(
        context=lambda x: format_docs(retriever.invoke(x["rewritten"]))
    )
    # 3. 원본 질문과 검색된 문서를 프롬프트에 넣는다.
    # --------------------------------------------------
    # 3단계. question과 context를 프롬프트에 삽입
    # --------------------------------------------------
    # answer_prompt는 {question}, {context} 값을 사용한다.
    # rewritten은 검색에는 사용되지만, 최종 프롬프트에는 직접 사용되지 않는다.
    | answer_prompt
    | llm
    | StrOutputParser()
)

result = rewrite_rag_chain.invoke({"question": "AI가 회사 일에 어떻게 들어와요?"})
print(result)


AI는 기업 운영 인프라로 자리 잡아가며, 의사결정, 운영, 고객 대응 등 주요 영역에 단계적으로 통합됩니다.  
기업 내부에서 AI 에이전트를 활용한 문서 처리, 고객 지원, 운영 자동화 등이 증가하고 있습니다.  
AI 활용이 여러 산업군에서 실험 단계를 넘어 일부 핵심 공정과 서비스 운영에 본격 도입되기 시작하고 있습니다.  
기업 간 시스템 연동을 통해 AI 간 협업 형태의 자동화 사례도 등장할 것으로 예상됩니다.


## 정리

| 방법 | 설명 | 효과 |
|---|---|---|
| Query Rewriting | LLM으로 구어체 질문을 보고서 검색에 적합한 질의로 변환 | 검색 정밀도 향상 |
| HyDE | 질문에 대한 가상 보고서 답변을 생성한 후 검색 | 질문과 문서 표현 간 의미 간격 축소 |
| Rewrite RAG Chain | 재작성된 질문으로 검색하고 검색 문서를 기반으로 답변 생성 | 모호한 질문에 대한 답변 안정성 향상 |

## 실습 확인 포인트

- 원본 질문과 재작성 질문의 검색 결과가 달라지는지 확인한다.
- 재작성 질문이 보고서의 실제 섹션 용어와 가까워졌는지 확인한다.
- HyDE로 생성한 가상 문서가 관련 섹션 검색에 도움이 되는지 확인한다.
- ChromaDB의 score는 거리값이므로 낮을수록 더 유사하다고 해석한다.
- PDF 직접 로딩 기반 DB와 Markdown 기반 DB의 score 절대값은 직접 비교하지 않는다.


## 주의할 점

- Query Rewriting은 원본 질문의 의미를 유지하면서 검색에 적합한 표현으로 바꾸는 것이 핵심이다.
- 재작성 과정에서 질문 의도가 바뀌면 검색 결과가 오히려 나빠질 수 있다.
- HyDE는 질문에 대한 가상의 답변을 생성해 검색하는 방식이므로, 생성된 문장을 최종 답변으로 해석하면 안 된다.
- HyDE 결과는 검색 품질을 높이기 위한 중간 표현이며, 최종 답변은 반드시 검색된 문서에 근거해야 한다.
